# Обучение Qwen2.5-Coder-3B на PAUQ через QLoRA

**Где запускать:** Kaggle Notebook с GPU T4 (Settings → Accelerator → GPU T4 x2 или T4 x1).

**Что делаем:**
1. Ставим зависимости.
2. Качаем PAUQ.
3. Загружаем Qwen2.5-Coder-3B в 4-bit.
4. Готовим датасет в chat-формате.
5. Дообучаем LoRA-адаптер через `SFTTrainer`.
6. Сохраняем адаптер локально и (опционально) пушим на HuggingFace Hub.

**Время на T4:** ~2–3 часа на эпоху (если ~10к примеров, max_seq_length=1024).

## 1. Установка зависимостей

In [ ]:
!pip install -q -U \
    transformers==4.44.2 \
    peft==0.12.0 \
    accelerate==0.33.0 \
    bitsandbytes==0.43.3 \
    trl==0.10.1 \
    datasets==2.20.0 \
    sqlglot==25.5.1 \
    wandb

## 2. Авторизация HuggingFace и W&B

В Kaggle добавь секреты: `HF_TOKEN` и `WANDB_API_KEY` через Add-ons → Secrets. Тогда они подхватятся автоматически.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

import wandb
wandb.login()

## 3. Скачиваем PAUQ

Альтернатива: загрузить PAUQ как Kaggle Dataset и подключить через `/kaggle/input/`.

In [ ]:
!git clone https://github.com/ai-forever/pauq.git /kaggle/working/pauq_repo
!ls /kaggle/working/pauq_repo

In [ ]:
import json
from pathlib import Path

# Точные пути зависят от структуры репозитория. Найди train/dev/test файлы:
for p in Path("/kaggle/working/pauq_repo").rglob("*.json"):
    print(p)

In [ ]:
# ОБНОВИ пути после `ls` выше
TRAIN_JSON = Path("/kaggle/working/pauq_repo/path/to/train.json")
DEV_JSON = Path("/kaggle/working/pauq_repo/path/to/dev.json")
DATABASES_DIR = Path("/kaggle/working/pauq_repo/path/to/databases")

with TRAIN_JSON.open() as f:
    train_raw = json.load(f)
with DEV_JSON.open() as f:
    dev_raw = json.load(f)

print(f"train: {len(train_raw)}, dev: {len(dev_raw)}")
print("Пример:", train_raw[0])

## 4. SchemaRetriever и PromptBuilder (инлайн)

В Kaggle нет нашего пакета `src/`, поэтому копируем минимум нужного кода прямо сюда.

In [ ]:
import sqlite3
from functools import lru_cache

SYSTEM_PROMPT = (
    "Ты — ассистент, который преобразует вопросы на русском языке в корректные SQL-запросы. "
    "Тебе даётся схема базы данных в виде CREATE TABLE statements и пример нескольких строк. "
    "Сгенерируй один SQL-запрос, который отвечает на вопрос пользователя. "
    "Возвращай ТОЛЬКО SQL без объяснений, без markdown, без префиксов."
)

@lru_cache(maxsize=512)
def render_schema(db_id: str, n_samples: int = 2) -> str:
    db_path = DATABASES_DIR / db_id / f"{db_id}.sqlite"
    if not db_path.exists():
        return ""
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    conn.text_factory = lambda b: b.decode("utf-8", errors="replace")
    cur = conn.cursor()
    cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
    parts = []
    for name, ddl in cur.fetchall():
        if not ddl:
            continue
        parts.append(ddl.strip() + ";")
        try:
            cur.execute(f'SELECT * FROM "{name}" LIMIT {n_samples}')
            rows = cur.fetchall()
            for r in rows:
                parts.append(f"-- {r}")
        except sqlite3.Error:
            pass
        parts.append("")
    conn.close()
    return "\n".join(parts).strip()

def build_messages(schema: str, question: str, sql: str | None = None):
    user = f"### Schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    if sql is not None:
        msgs.append({"role": "assistant", "content": sql.strip()})
    return msgs

## 5. Готовим датасет для SFT

In [ ]:
from datasets import Dataset

def to_record(item):
    q = item.get("question") or item.get("question_ru") or ""
    sql = item.get("query") or item.get("sql_query") or item.get("sql") or ""
    db_id = item.get("db_id") or item.get("database") or ""
    if not (q and sql and db_id):
        return None
    schema = render_schema(db_id)
    if not schema:
        return None
    return {"messages": build_messages(schema, q.strip(), sql.strip())}

train_records = [r for r in (to_record(x) for x in train_raw) if r]
dev_records = [r for r in (to_record(x) for x in dev_raw) if r]
print(f"train usable: {len(train_records)}, dev usable: {len(dev_records)}")

train_ds = Dataset.from_list(train_records)
dev_ds = Dataset.from_list(dev_records)

## 6. Загружаем модель в 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

## 7. Конфиг LoRA

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

## 8. Тренировка через SFTTrainer

In [ ]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "/kaggle/working/qwen-coder-pauq-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    bf16=True,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="no",  # eval делаем отдельно после тренировки
    max_seq_length=1024,
    packing=False,
    report_to="wandb",
    run_name="qwen3b-pauq-qlora",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    peft_config=lora_config,
    args=sft_config,
)

trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)

## 9. Быстрая проверка inference

In [ ]:
model.config.use_cache = True
model.eval()

ex = dev_records[0]
prompt_msgs = ex["messages"][:2]  # без assistant-ответа
prompt = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
new_tokens = out[0][inputs["input_ids"].shape[1]:]
print("Pred:", tokenizer.decode(new_tokens, skip_special_tokens=True))
print("Gold:", ex["messages"][2]["content"])

## 10. Загрузка адаптера на HuggingFace Hub (приватный репо)

In [ ]:
HF_REPO = "your-username/qwen-coder-pauq-lora"  # замени на свой

trainer.model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)
print("Pushed to", HF_REPO)

## Дальше

1. Скачай адаптер на десктоп: `huggingface-cli download your-username/qwen-coder-pauq-lora --local-dir checkpoints/qwen-coder-pauq-lora`.
2. Запусти `python -m src.evaluation.evaluate --split dev --limit 100` локально, либо запусти полный eval здесь же на Kaggle.
3. Если метрики низкие: проверь prompt format, увеличь эпохи, понизь learning rate.